# Water Quality Prediction using Bayesian Belief Network
## Assignment 2 - PS10

### Overview
This notebook implements a Bayesian Belief Network (BBN) for water quality prediction, addressing all four questions in the assignment:
1. Construct a Bayesian Belief Network for water quality data
2. Predict water potability for given attribute values
3. Infer probability for given attribute values including potability
4. Find conditional probability of water quality being good under specific conditions

In [1]:
#!pip install pgmpy

In [2]:
# Import required libraries
import os
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator, MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
import warnings
warnings.filterwarnings('ignore')


C:\Users\Dell\anaconda3\Lib\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


## WaterQualityBBN Class Definition

In [3]:
class WaterQualityBBN:
    """
    Bayesian Belief Network for Water Quality Prediction
    """

    def __init__(self, data_file=None):
        self.data = None
        self.model = None
        self.infer = None
        self.discretized_data = None
        # Stores qcut bin edges per column so query-time categorization
        # uses the identical boundaries as training-time discretization.
        self._bin_edges = {}
        if data_file:
            self.load_data(data_file)

    def load_data(self, data_file):
        """Load water quality data from CSV file."""
        try:
            self.data = pd.read_csv(data_file)
            print(f"Data loaded successfully. Shape: {self.data.shape}")
            print(f"Columns: {list(self.data.columns)}")
        except FileNotFoundError:
            raise FileNotFoundError(f"Data file {data_file} not found")
        except Exception as e:
            raise Exception(f"Error loading data: {str(e)}")

    def discretize_data(self):
        """
        Discretize continuous variables into categories (low, medium, high).
        Stores exact qcut bin edges in self._bin_edges for consistent query-time use.
        """
        if self.data is None:
            raise ValueError("No data loaded. Please load data first.")

        self.discretized_data = self.data.copy()

        continuous_cols = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate',
                           'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']

        for col in continuous_cols:
            if col in self.discretized_data.columns:
                # retbins=True captures the exact bin edges for consistent query-time use
                discretized_col, bins = pd.qcut(
                    self.discretized_data[col],
                    q=3,
                    labels=['low', 'medium', 'high'],
                    duplicates='drop',
                    retbins=True
                )
                self.discretized_data[col] = discretized_col
                self._bin_edges[col] = bins  # [min, q1_edge, q2_edge, max]

        # Drop rows with NaN values (pgmpy cannot handle missing values)
        self.discretized_data.dropna(inplace=True)
        self.discretized_data.reset_index(drop=True, inplace=True)

        # Ensure Potability stays as integer after dropna (avoids float drift)
        self.discretized_data['Potability'] = self.discretized_data['Potability'].astype(int)

        print("Data discretized into categories: low, medium, high")
        print(f"Discretized data shape after dropping NaN: {self.discretized_data.shape}")

    def construct_bbn(self):
        """
        Construct Bayesian Belief Network structure.

        Uses a GENERATIVE Naïve Bayes topology:  Potability → each attribute

        Why: the discriminative direction (attributes → Potability) creates a
        joint CPD for Potability with 3^9 × 2 = 39,366 parameters.  With ~2011
        training rows almost every cell is empty and Dirichlet smoothing drives
        all predictions to ~0.5 (dummy outputs).

        Flipping to the generative direction yields:
            P(Potability)            — 2 parameters
            P(attribute|Potability)  — 3×2 = 6 parameters × 9 attributes = 54
        Total: 56 parameters — tractable and data-driven.

        Inference: Variable Elimination computes P(Potability | observed
        attributes) via Bayes' rule, giving the same query as the discriminative
        direction but with reliable, non-uniform probabilities.
        """
        if self.discretized_data is None:
            self.discretize_data()

        attribute_nodes = [
            'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate',
            'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity'
        ]

        # Generative direction: Potability is the root/parent of every attribute
        edges = [('Potability', attr) for attr in attribute_nodes
                 if attr in self.discretized_data.columns]

        self.model = DiscreteBayesianNetwork(edges)
        print(f"Bayesian Network structure defined ({len(edges)} edges, "
              f"generative Naïve Bayes topology)")

    def learn_parameters(self):
        """
        Learn CPD parameters from data.

        Cascade to handle different pgmpy versions:
          1. BayesianEstimator.get_parameters() + add_cpds()  — version-agnostic
          2. MaximumLikelihoodEstimator(model, data) as instance — new API
          3. DiscreteMLE() — latest pgmpy builds
          4. MaximumLikelihoodEstimator class (uninstantiated) — old API
        """
        if self.model is None:
            self.construct_bbn()

        errors = []

        # Attempt 1: BayesianEstimator.get_parameters() → add_cpds()
        # Bypasses fit() entirely; works across all pgmpy versions.
        try:
            est = BayesianEstimator(self.model, self.discretized_data)
            cpds = est.get_parameters(prior_type='dirichlet', pseudo_counts=1)
            self.model.add_cpds(*cpds)
            if self.model.check_model():
                print("Parameters learned successfully (BayesianEstimator.get_parameters)")
                print("Model validation passed — CPDs are consistent")
            self.infer = VariableElimination(self.model)
            return
        except Exception as e:
            errors.append(f"BayesianEstimator.get_parameters: {e}")

        # Attempt 2: MaximumLikelihoodEstimator as an initialized instance (new API)
        try:
            est = MaximumLikelihoodEstimator(self.model, self.discretized_data)
            self.model.fit(self.discretized_data, estimator=est)
            print("Parameters learned successfully (MaximumLikelihoodEstimator instance)")
            self.infer = VariableElimination(self.model)
            return
        except Exception as e:
            errors.append(f"MaximumLikelihoodEstimator instance: {e}")

        # Attempt 3: DiscreteMLE() — latest pgmpy builds
        try:
            from pgmpy.estimators import DiscreteMLE
            self.model.fit(self.discretized_data, estimator=DiscreteMLE())
            print("Parameters learned successfully (DiscreteMLE)")
            self.infer = VariableElimination(self.model)
            return
        except Exception as e:
            errors.append(f"DiscreteMLE: {e}")

        # Attempt 4: Uninstantiated class — old pgmpy API (< 0.1.24)
        try:
            self.model.fit(self.discretized_data, estimator=MaximumLikelihoodEstimator)
            print("Parameters learned successfully (old MLE class API)")
            self.infer = VariableElimination(self.model)
            return
        except Exception as e:
            errors.append(f"old MLE class: {e}")

        raise Exception(f"Failed to learn parameters. Tried: {'; '.join(errors)}")

    def get_category(self, value, column):
        """
        Map a continuous value to its discretized category (low/medium/high).

        Uses the exact bin edges captured by pd.qcut during training so that
        query-time categorization is identical to training-time discretization.
        Falls back to raw percentile calculation if edges were not stored.
        """
        if self.data is None:
            raise ValueError("No data loaded")

        if column in self._bin_edges:
            bins = self._bin_edges[column]  # [min_edge, q1_edge, q2_edge, max_edge]
            if value <= bins[1]:
                return 'low'
            elif value <= bins[2]:
                return 'medium'
            else:
                return 'high'
        else:
            col_data = self.data[column].dropna()
            q1, q2 = col_data.quantile([0.3333, 0.6667])
            if value <= q1:
                return 'low'
            elif value <= q2:
                return 'medium'
            else:
                return 'high'

    def predict_potability(self, input_dict):
        """Predict water potability for given input values (Question 2)."""
        if self.infer is None:
            self.learn_parameters()

        evidence = {
            key: self.get_category(value, key)
            for key, value in input_dict.items()
            if key in self.data.columns and key != 'Potability'
        }

        try:
            result = self.infer.query(variables=['Potability'], evidence=evidence)
            prob_0 = result.values[0]
            prob_1 = result.values[1]
            return {'prediction': int(prob_1 > prob_0), 'prob_0': prob_0, 'prob_1': prob_1}
        except Exception as e:
            print(f"Error during prediction: {str(e)}")
            return {'prediction': 0, 'prob_0': 0.5, 'prob_1': 0.5}

    def infer_probability(self, input_dict):
        """Infer P(Potability | observed attributes) — Question 3."""
        if self.infer is None:
            self.learn_parameters()

        evidence = {
            key: self.get_category(value, key)
            for key, value in input_dict.items()
            if key in self.data.columns and key != 'Potability'
        }

        try:
            result = self.infer.query(variables=['Potability'], evidence=evidence)
            return {'prob_0': result.values[0], 'prob_1': result.values[1]}
        except Exception as e:
            print(f"Error during inference: {str(e)}")
            return {'prob_0': 0.5, 'prob_1': 0.5}

    def infer_conditional_probability(self, conditions):
        """
        Find P(Potability | categorical conditions) — Question 4.
        conditions: dict of {attribute: 'low'|'medium'|'high'}
        """
        if self.infer is None:
            self.learn_parameters()

        evidence = {
            key: value
            for key, value in conditions.items()
            if key in self.discretized_data.columns and key != 'Potability'
        }

        try:
            result = self.infer.query(variables=['Potability'], evidence=evidence)
            prob_0 = result.values[0]
            prob_1 = result.values[1]
            return {'prob_0': prob_0, 'prob_1': prob_1, 'percentage': prob_1 * 100}
        except Exception as e:
            print(f"Error during conditional inference: {str(e)}")
            return {'prob_0': 0.5, 'prob_1': 0.5, 'percentage': 50.0}


## Helper Functions for File I/O

In [4]:
def read_input_file(input_file):
    """
    Read input from inputPS10.txt — one 'key value' pair per line.
    Lines starting with '#' are treated as comments.
    """
    with open(input_file, 'r') as f:
        lines = f.readlines()

    input_data = {}
    for line in lines:
        line = line.strip()
        if line and not line.startswith('#'):
            parts = line.split()
            if len(parts) >= 2:
                key = parts[0]
                try:
                    input_data[key] = float(parts[1])
                except ValueError:
                    input_data[key] = parts[1]

    return input_data


def append_output_file(output_file, results, query_type):
    """Append a labelled query result block to outputPS10.txt."""
    section_labels = {
        'prediction':  'Question 2: Water Quality Prediction',
        'inference':   'Question 3: Probability Inference',
        'conditional': 'Question 4: Conditional Probability (low ph, high hardness, high solids)',
    }
    with open(output_file, 'a') as f:
        f.write(f"--- {section_labels.get(query_type, query_type)} ---\n")
        _write_query_result(f, results, query_type)
        f.write("\n")


def _write_query_result(f, results, query_type):
    """Write a single query result block to file handle f."""
    if query_type == 'prediction':
        f.write("Potability 0 1\n")
        f.write(f"Probability {results['prob_0']:.6f} {results['prob_1']:.6f}\n")
        f.write(f"Probability of the water being good is {results['prob_1']*100:.2f}% when considered low ph value and remaining variables high value from given dataset\n")

    elif query_type == 'inference':
        f.write("Potability 0 1\n")
        f.write(f"Probability {results['prob_0']:.5f} {results['prob_1']:.5f}\n")
        f.write(f"Probability of the water being good is {results['prob_1']*100:.2f}% when the given attribute values are given from the dataset\n")

    elif query_type == 'conditional':
        f.write("Potability 0 1\n")
        f.write(f"Probability {results['prob_0']:.6f} {results['prob_1']:.6f}\n")
        f.write(f"Probability of the water being good is {results['percentage']:.2f}% when considered low ph value and remaining variables high value from given dataset\n")


## Main Execution

In [5]:
# Configuration
data_file = 'water_potability.csv'  # Water potability dataset
input_file = 'inputPS10.txt'
output_file = 'outputPS10.txt'

print("Water Quality Prediction using Bayesian Belief Network")
print("=" * 60)


Water Quality Prediction using Bayesian Belief Network


### Initialize and Train the BBN

In [6]:
# Initialize BBN
try:
    bbn = WaterQualityBBN(data_file)
    bbn.discretize_data()
    bbn.construct_bbn()
    bbn.learn_parameters()
    print("\nBayesian Belief Network constructed and trained successfully\n")
except Exception as e:
    print(f"\nError: {str(e)}")
    print("\nPlease ensure 'water_potability.csv' exists in the directory")
    print("The CSV file should contain columns: ph, Hardness, Solids, Chloramines,")
    print("Sulfate, Conductivity, Organic_carbon, Trihalomethanes, Turbidity, Potability")


Data loaded successfully. Shape: (3276, 10)
Columns: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability']
Data discretized into categories: low, medium, high
Discretized data shape after dropping NaN: (2011, 10)
Bayesian Network structure defined (9 edges, generative Naïve Bayes topology)
Parameters learned successfully (BayesianEstimator.get_parameters)
Model validation passed — CPDs are consistent

Bayesian Belief Network constructed and trained successfully



### Read Input and Perform Query

In [7]:
# Read input file for Question 2 data
try:
    input_data = read_input_file(input_file)
    print(f"Input data read from {input_file}")
    print(f"Input: {input_data}\n")
except FileNotFoundError:
    print(f"Input file {input_file} not found. Using sample data for demonstration.\n")
    input_data = {
        'ph': 3.72,
        'Hardness': 204.89,
        'Solids': 20791.32,
        'Chloramines': 7.3,
        'Sulfate': 368.5,
        'Conductivity': 564.30,
        'Turbidity': 2.96
    }

# Delete existing output file to ensure a clean run
if os.path.exists(output_file):
    os.remove(output_file)
    print(f"Previous {output_file} removed.")

# Create fresh output file
with open(output_file, 'w') as f:
    f.write("Water Quality Prediction using Bayesian Belief Network\n")
    f.write("=" * 60 + "\n\n")

# --- Question 2: Predict water quality ---
print("Question 2: Performing potability prediction...")
q2_input = {k: v for k, v in input_data.items() if k != 'Potability'}
results_q2 = bbn.predict_potability(q2_input)
append_output_file(output_file, results_q2, 'prediction')
print(f"Q2 -> Potability 0: {results_q2['prob_0']:.6f}  1: {results_q2['prob_1']:.6f}")

# --- Question 3: Infer probability ---
print("\nQuestion 3: Performing probability inference...")
q3_input = {
    'Hardness':        input_data.get('Hardness', 248.0),
    'Solids':          input_data.get('Solids', 28749),
    'Chloramines':     input_data.get('Chloramines', 7.5),
    'Sulfate':         input_data.get('Sulfate', 393),
    'Conductivity':    input_data.get('Conductivity', 283),
    'Organic_carbon':  input_data.get('Organic_carbon', 13.78),
    'Trihalomethanes': input_data.get('Trihalomethanes', 84.6),
    'Turbidity':       input_data.get('Turbidity', 2.67),
}
results_q3 = bbn.infer_probability(q3_input)
append_output_file(output_file, results_q3, 'inference')
print(f"Q3 -> Potability 0: {results_q3['prob_0']:.5f}  1: {results_q3['prob_1']:.5f}")

# --- Question 4: Conditional probability (low ph, high hardness, high solids) ---
print("\nQuestion 4: Performing conditional probability inference...")
conditions_q4 = {
    'ph':       'low',
    'Hardness': 'high',
    'Solids':   'high'
}
results_q4 = bbn.infer_conditional_probability(conditions_q4)
append_output_file(output_file, results_q4, 'conditional')
print(f"Q4 -> Potability 0: {results_q4['prob_0']:.6f}  1: {results_q4['prob_1']:.6f}")

print(f"\nAll results written to {output_file}")


Input file inputPS10.txt not found. Using sample data for demonstration.

Previous outputPS10.txt removed.
Question 2: Performing potability prediction...
Q2 -> Potability 0: 0.712958  1: 0.287042

Question 3: Performing probability inference...
Q3 -> Potability 0: 0.67632  1: 0.32368

Question 4: Performing conditional probability inference...
Q4 -> Potability 0: 0.547275  1: 0.452725

All results written to outputPS10.txt


### Display Output

In [8]:
print("\nOutput:")
with open(output_file, 'r') as f:
    print(f.read())



Output:
Water Quality Prediction using Bayesian Belief Network

--- Question 2: Water Quality Prediction ---
Potability 0 1
Probability 0.712958 0.287042
Probability of the water being good is 28.70% when considered low ph value and remaining variables high value from given dataset

--- Question 3: Probability Inference ---
Potability 0 1
Probability 0.67632 0.32368
Probability of the water being good is 32.37% when the given attribute values are given from the dataset

--- Question 4: Conditional Probability (low ph, high hardness, high solids) ---
Potability 0 1
Probability 0.547275 0.452725
Probability of the water being good is 45.27% when considered low ph value and remaining variables high value from given dataset




## Example: Question 2 - Prediction

In [9]:
# Example for Question 2: Predict water quality
prediction_input = {
    'ph': 3.72,
    'Hardness': 204.89,
    'Solids': 20791.32,
    'Chloramines': 7.3,
    'Sulfate': 368.5,
    'Conductivity': 564.30,
    'Turbidity': 2.96
}

results = bbn.predict_potability(prediction_input)
print(f"Prediction Results:")
print(f"Potability 0 1")
print(f"Probability {results['prob_0']:.6f} {results['prob_1']:.6f}")
print(f"Predicted Potability: {results['prediction']}")


Prediction Results:
Potability 0 1
Probability 0.712958 0.287042
Predicted Potability: 0


## Example: Question 3 - Inference

In [10]:
# Example for Question 3: Infer probability
inference_input = {
    'Hardness': 248.0,
    'Solids': 28749,
    'Chloramines': 7.5,
    'Sulfate': 393,
    'Conductivity': 283,
    'Organic_carbon': 13.78,
    'Trihalomethanes': 84.6,
    'Turbidity': 2.67,
    'Potability': 1
}

results = bbn.infer_probability(inference_input)
print(f"Inference Results:")
print(f"Potability 0 1")
print(f"Probability {results['prob_0']:.5f} {results['prob_1']:.5f}")
print(f"Probability of the water being good is {results['prob_1']*100:.2f}% when the given attribute values are given from the dataset")


Inference Results:
Potability 0 1
Probability 0.49209 0.50791
Probability of the water being good is 50.79% when the given attribute values are given from the dataset


## Example: Question 4 - Conditional Probability

In [11]:
# Example for Question 4: Conditional probability
conditions = {
    'ph': 'low',
    'Hardness': 'high',
    'Solids': 'high'
}

results = bbn.infer_conditional_probability(conditions)
print(f"Conditional Probability Results:")
print(f"Potability 0 1")
print(f"Probability {results['prob_0']:.6f} {results['prob_1']:.6f}")
print(f"Probability of the water being good is {results['percentage']:.2f}% when considered low ph value and remaining variables high value from given dataset")


Conditional Probability Results:
Potability 0 1
Probability 0.547275 0.452725
Probability of the water being good is 45.27% when considered low ph value and remaining variables high value from given dataset


In [21]:
import pypandoc

pypandoc.convert_file(
    'designPS10_group90.md',
    'pdf',
    outputfile='designPS10_group90.pdf',
    extra_args=['--standalone']
)

print("PDF created successfully!")

RuntimeError: Pandoc died with exitcode "47" during conversion: pdflatex not found. Please select a different --pdf-engine or install pdflatex

Hint: pytinytex is installed but could not resolve the missing LaTeX packages. You may need to install them manually with pytinytex.install('<package>').